In [88]:
import json
import time
from datetime import datetime
from http.client import IncompleteRead, RemoteDisconnected
from pprint import pprint
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import urlopen

from pymongo import MongoClient, ASCENDING, GEOSPHERE, TEXT, UpdateOne


In [89]:
SOCRATA_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
MONGO_URI = "mongodb://localhost:27017"

DB_NAME = "ds4300_nyc311"
COLLECTION_NAME = "service_requests"

In [ ]:
START_DATE = "2021-01-01T00:00:00"
END_DATE = "2025-12-28T00:00:00"

PAGE_SIZE = 10_000
MAX_ROWS = 21_456_551


In [91]:
FIELDS = [
    "unique_key", "created_date", "closed_date", "agency", "agency_name",
    "complaint_type", "descriptor", "status", "borough", "incident_zip",
    "latitude", "longitude", "location", "resolution_description"
]

In [92]:
def get_data(params, retries=4):
    url = SOCRATA_URL + "?" + urlencode(params)

    for attempt in range(retries):
        try:
            with urlopen(url, timeout=90) as response:
                return json.load(response)
        except (IncompleteRead, RemoteDisconnected, TimeoutError, URLError, ConnectionResetError) as error:
            if attempt == retries - 1:
                raise
            print(f"Retrying after {type(error).__name__}")
            time.sleep(2 ** attempt)


In [93]:
where = f"created_date >= '{START_DATE}' AND created_date < '{END_DATE}'"

base_query = {
    "$select": ",".join(FIELDS),
    "$where": where,
    "$order": "created_date"
}

In [94]:
sample = get_data(base_query | {"$limit": 5})
len(sample)

5

In [95]:
pprint(sample[0])

{'agency': 'NYPD',
 'agency_name': 'New York City Police Department',
 'borough': 'BRONX',
 'closed_date': '2025-01-02T17:05:54.000',
 'complaint_type': 'Noise - Residential',
 'created_date': '2025-01-01T00:00:12.000',
 'descriptor': 'Loud Music/Party',
 'incident_zip': '10466',
 'latitude': '40.89187241649303',
 'location': {'coordinates': [-73.860168452965, 40.891872416493],
              'type': 'Point'},
 'longitude': '-73.86016845296459',
 'resolution_description': 'The Police Department responded to the complaint '
                           'and determined that police action was not '
                           'necessary.',
 'status': 'Closed',
 'unique_key': '63577994'}


In [96]:
def clean(row):
    doc = dict(row)

    doc["_id"] = row["unique_key"]
    doc["created_dt"] = datetime.fromisoformat(row["created_date"]) if row.get("created_date") else None
    doc["closed_dt"] = datetime.fromisoformat(row["closed_date"]) if row.get("closed_date") else None

    doc["lat"] = float(row["latitude"]) if row.get("latitude") else None
    doc["lon"] = float(row["longitude"]) if row.get("longitude") else None

    return doc

In [97]:
def add_features(doc):
    if doc["lat"] is not None and doc["lon"] is not None:
        doc["location"] = {"type": "Point", "coordinates": [doc["lon"], doc["lat"]]}

    if doc["created_dt"] and doc["closed_dt"]:
        hours = (doc["closed_dt"] - doc["created_dt"]).total_seconds() / 3600
        doc["resolution_hours"] = round(hours, 2)

    return doc

In [98]:
def transform(row):
    return add_features(clean(row))

In [99]:
sample_docs = [transform(row) for row in sample]
sample_docs[0]

{'unique_key': '63577994',
 'created_date': '2025-01-01T00:00:12.000',
 'closed_date': '2025-01-02T17:05:54.000',
 'agency': 'NYPD',
 'agency_name': 'New York City Police Department',
 'complaint_type': 'Noise - Residential',
 'descriptor': 'Loud Music/Party',
 'status': 'Closed',
 'borough': 'BRONX',
 'incident_zip': '10466',
 'latitude': '40.89187241649303',
 'longitude': '-73.86016845296459',
 'location': {'type': 'Point',
  'coordinates': [-73.86016845296459, 40.89187241649303]},
 'resolution_description': 'The Police Department responded to the complaint and determined that police action was not necessary.',
 '_id': '63577994',
 'created_dt': datetime.datetime(2025, 1, 1, 0, 0, 12),
 'closed_dt': datetime.datetime(2025, 1, 2, 17, 5, 54),
 'lat': 40.89187241649303,
 'lon': -73.86016845296459,
 'resolution_hours': 41.09}

In [100]:
client = MongoClient(MONGO_URI)
client.admin.command("ping")

{'ok': 1.0}

In [101]:
db = client[DB_NAME]
collection = db[COLLECTION_NAME]
collection.name

'service_requests'

In [102]:
collection.create_index([("created_dt", ASCENDING)])
collection.create_index([("borough", ASCENDING)])
collection.create_index([("location", GEOSPHERE)])

'location_2dsphere'

In [103]:
collection.create_index([
    ("complaint_type", TEXT),
    ("descriptor", TEXT),
    ("resolution_description", TEXT)
])

'complaint_type_text_descriptor_text_resolution_description_text'

In [104]:
ops = [UpdateOne({"_id": doc["_id"]}, {"$set": doc}, upsert=True) for doc in sample_docs]
collection.bulk_write(ops, ordered=False)

BulkWriteResult({'writeErrors': [], 'writeConcernErrors': [], 'nInserted': 0, 'nUpserted': 0, 'nMatched': 5, 'nModified': 0, 'nRemoved': 0, 'upserted': []}, acknowledged=True)

In [105]:
collection.count_documents({})

50000

In [106]:
page = get_data(base_query | {"$limit": PAGE_SIZE, "$offset": 0})
len(page)


10000

In [107]:
for offset in range(0, MAX_ROWS, PAGE_SIZE):
    page = get_data(base_query | {"$limit": PAGE_SIZE, "$offset": offset})
    if not page:
        break

    docs = [transform(row) for row in page]
    ops = [UpdateOne({"_id": doc["_id"]}, {"$set": doc}, upsert=True) for doc in docs]
    collection.bulk_write(ops, ordered=False)

    print(offset + len(page), collection.count_documents({}))
    time.sleep(0.5)


10000 50000
20000 50000
30000 50000
40000 50000
50000 50000
60000 60000
70000 70000
80000 80000
90000 90000
100000 100000
110000 110000
120000 120000
130000 130000
140000 140000
150000 150000
160000 160000
170000 170000
180000 180000
190000 190000
200000 200000
210000 210000
220000 220000
230000 230000
Retrying after URLError
240000 240000
250000 250000
260000 260000
270000 270000
280000 280000
290000 290000
300000 300000
310000 310000
320000 320000
330000 330000
340000 340000
350000 350000
360000 360000
370000 370000
380000 380000
390000 390000
400000 400000
410000 410000
420000 420000
430000 430000
440000 440000
450000 450000
460000 460000
470000 470000
480000 480000
490000 490000
500000 500000
510000 510000
520000 520000
530000 530000
540000 540000
550000 550000
560000 560000
570000 570000
580000 580000
590000 590000
594750 594750


In [108]:
collection.count_documents({})


594750

In [109]:
list(collection.find({}, {
    "_id": 1, "borough": 1, "complaint_type": 1, "resolution_hours": 1
}).limit(5))


[{'_id': '63577994',
  'borough': 'BRONX',
  'complaint_type': 'Noise - Residential',
  'resolution_hours': 41.09},
 {'_id': '63584096',
  'borough': 'BROOKLYN',
  'complaint_type': 'Noise - Residential',
  'resolution_hours': 0.09},
 {'_id': '63581480',
  'borough': 'BRONX',
  'complaint_type': 'Noise - Residential',
  'resolution_hours': 9.01},
 {'_id': '63575612',
  'borough': 'BROOKLYN',
  'complaint_type': 'Illegal Fireworks',
  'resolution_hours': 0.71},
 {'_id': '63586536',
  'borough': 'BRONX',
  'complaint_type': 'HEAT/HOT WATER',
  'resolution_hours': 36.07}]

In [110]:
# Average Resolution Time By Borough
pipeline = [
    {"$match": {"resolution_hours": {"$gte": 0, "$lte": 720}}},
    {"$group": {"_id": "$borough", "count": {"$sum": 1}, "avg_hours": {"$avg": "$resolution_hours"}}},
    {"$sort": {"avg_hours": 1}}
]


In [111]:
list(collection.aggregate(pipeline))

[{'_id': 'QUEENS', 'count': 114854, 'avg_hours': 39.33693271457677},
 {'_id': 'BRONX', 'count': 161378, 'avg_hours': 48.402713938702924},
 {'_id': 'STATEN ISLAND', 'count': 17532, 'avg_hours': 51.51578770248688},
 {'_id': 'BROOKLYN', 'count': 153779, 'avg_hours': 54.29557052653483},
 {'_id': 'MANHATTAN', 'count': 94899, 'avg_hours': 56.84649490510964},
 {'_id': 'Unspecified', 'count': 250, 'avg_hours': 90.44475999999999}]

In [112]:
# Top Complaint Types
pipeline = [
    {"$group": {"_id": "$complaint_type", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 10}
]

In [113]:
list(collection.aggregate(pipeline))

[{'_id': 'HEAT/HOT WATER', 'count': 107149},
 {'_id': 'Noise - Residential', 'count': 97009},
 {'_id': 'Illegal Parking', 'count': 83674},
 {'_id': 'Blocked Driveway', 'count': 26722},
 {'_id': 'UNSANITARY CONDITION', 'count': 17300},
 {'_id': 'PLUMBING', 'count': 12941},
 {'_id': 'Water System', 'count': 11737},
 {'_id': 'Street Condition', 'count': 10927},
 {'_id': 'Abandoned Vehicle', 'count': 10313},
 {'_id': 'Traffic Signal Condition', 'count': 9910}]

In [114]:
# Fastest ZIP Codes
pipeline = [
    {"$match": {"resolution_hours": {"$gte": 0, "$lte": 720}, "incident_zip": {"$exists": True}}},
    {"$group": {"_id": "$incident_zip", "count": {"$sum": 1}, "avg_hours": {"$avg": "$resolution_hours"}}},
    {"$match": {"count": {"$gte": 50}}},
    {"$sort": {"avg_hours": 1}},
    {"$limit": 10}
]


In [115]:
list(collection.aggregate(pipeline))

[{'_id': '10169', 'count': 51, 'avg_hours': 10.020588235294118},
 {'_id': '11366', 'count': 3883, 'avg_hours': 10.922776203966006},
 {'_id': '10004', 'count': 979, 'avg_hours': 12.015454545454544},
 {'_id': '11239', 'count': 2995, 'avg_hours': 13.004971619365609},
 {'_id': '10466', 'count': 55113, 'avg_hours': 13.712790811605249},
 {'_id': '11421', 'count': 3129, 'avg_hours': 17.085155001597954},
 {'_id': '11378', 'count': 3254, 'avg_hours': 21.002274124154884},
 {'_id': '11101', 'count': 5140, 'avg_hours': 23.36768482490272},
 {'_id': '10282', 'count': 56, 'avg_hours': 25.46410714285714},
 {'_id': '10038', 'count': 887, 'avg_hours': 26.711736189402483}]

In [116]:
pipeline = [
    {"$match": {"resolution_hours": {"$gte": 0, "$lte": 720}}},
    {"$group": {
        "_id": {"borough": "$borough", "type": "$complaint_type"},
        "count": {"$sum": 1},
        "avg_hours": {"$avg": "$resolution_hours"}
    }},
    {"$match": {"count": {"$gte": 50}}},
    {"$sort": {"avg_hours": -1}},
    {"$limit": 20}
]

list(collection.aggregate(pipeline))

[{'_id': {'borough': 'MANHATTAN', 'type': 'For Hire Vehicle Complaint'},
  'count': 109,
  'avg_hours': 427.734495412844},
 {'_id': {'borough': 'BROOKLYN', 'type': 'Root/Sewer/Sidewalk Condition'},
  'count': 160,
  'avg_hours': 367.7064375},
 {'_id': {'borough': 'BROOKLYN', 'type': 'Maintenance or Facility'},
  'count': 424,
  'avg_hours': 358.7616037735849},
 {'_id': {'borough': 'QUEENS', 'type': 'School Maintenance'},
  'count': 65,
  'avg_hours': 356.05953846153847},
 {'_id': {'borough': 'MANHATTAN', 'type': 'Boilers'},
  'count': 227,
  'avg_hours': 347.92211453744494},
 {'_id': {'borough': 'BRONX', 'type': 'Boilers'},
  'count': 210,
  'avg_hours': 335.65319047619045},
 {'_id': {'borough': 'BRONX', 'type': 'Graffiti'},
  'count': 101,
  'avg_hours': 317.089405940594},
 {'_id': {'borough': 'BRONX', 'type': 'ELEVATOR'},
  'count': 60,
  'avg_hours': 316.08983333333333},
 {'_id': {'borough': 'QUEENS', 'type': 'Boilers'},
  'count': 146,
  'avg_hours': 310.96205479452055},
 {'_id': {